In [1]:
import sys
from pathlib import Path

ROOT = Path(r"C:\____Moje-MOJE\MyProjects_4Fun\projects\World of Warcraft\python-etl")
sys.path.insert(0, str(ROOT))

In [2]:
from moduly.ai_przyklady_ras_tony_teksty import RACE_STYLES
from sqlalchemy import text

In [3]:
from moduly.db_core import utworz_engine_do_db
silnik = utworz_engine_do_db()

In [4]:
q_select_npc = text("""
WITH wszystkie_idki AS (
    SELECT tabela_wartosci.ID_NPC
    FROM dbo.MISJE AS m
    CROSS APPLY (VALUES (m.NPC_START_ID), (m.NPC_KONIEC_ID)) AS tabela_wartosci (ID_NPC)
    WHERE m.MISJA_ID_MOJE_PK = :misja_id

    UNION

    SELECT ds.NPC_ID_FK
    FROM dbo.DIALOGI_STATUSY AS ds
    WHERE ds.MISJA_ID_MOJE_FK = :misja_id
),
oczyszczone_dane AS (
    SELECT wi.ID_NPC, ns.STATUS,
    CASE WHEN CHARINDEX('[', ns.NAZWA) > 0 THEN RTRIM(LEFT(ns.NAZWA, CHARINDEX('[', ns.NAZWA) - 1)) ELSE ns.NAZWA END AS CZYSTA_NAZWA
    FROM wszystkie_idki AS wi
    INNER JOIN dbo.NPC_STATUSY AS ns ON wi.ID_NPC = ns.NPC_ID_FK
)
    SELECT DISTINCT
        pvt.[0_ORYGINAŁ],
        pvt.[3_ZATWIERDZONO],
        n.PLEC,
        n.RASA
    FROM oczyszczone_dane
    PIVOT (MAX(CZYSTA_NAZWA) FOR STATUS IN ([0_ORYGINAŁ], [3_ZATWIERDZONO])) AS pvt
    LEFT JOIN dbo.NPC AS n
        ON n.NPC_ID_MOJE_PK = pvt.ID_NPC;
""")

In [13]:
q_select_rasa = text("""
WITH teksty_npc AS (
    -- NPC start: wszystkie segmenty oprócz zakończenia
    SELECT
        m.NPC_START_ID AS NPC_ID,
        SUM(LEN(ISNULL(ms.TRESC, N''))) AS ILE_ZNAKOW
    FROM dbo.MISJE_STATUSY AS ms
    INNER JOIN dbo.MISJE AS m
        ON ms.MISJA_ID_MOJE_FK = m.MISJA_ID_MOJE_PK
    WHERE ms.MISJA_ID_MOJE_FK = :misja_id
      AND ms.STATUS = N'0_ORYGINAŁ'
      AND ms.SEGMENT <> N'ZAKOŃCZENIE'
      AND m.NPC_START_ID IS NOT NULL
    GROUP BY m.NPC_START_ID

    UNION ALL

    -- NPC koniec: tylko zakończenie
    SELECT
        m.NPC_KONIEC_ID AS NPC_ID,
        SUM(LEN(ISNULL(ms.TRESC, N''))) AS ILE_ZNAKOW
    FROM dbo.MISJE_STATUSY AS ms
    INNER JOIN dbo.MISJE AS m
        ON ms.MISJA_ID_MOJE_FK = m.MISJA_ID_MOJE_PK
    WHERE ms.MISJA_ID_MOJE_FK = :misja_id
      AND ms.STATUS = N'0_ORYGINAŁ'
      AND ms.SEGMENT = N'ZAKOŃCZENIE'
      AND m.NPC_KONIEC_ID IS NOT NULL
    GROUP BY m.NPC_KONIEC_ID

    UNION ALL

    -- Dialogi: NPC per wypowiedź
    SELECT
        ds.NPC_ID_FK AS NPC_ID,
        SUM(LEN(ISNULL(ds.TRESC, N''))) AS ILE_ZNAKOW
    FROM dbo.DIALOGI_STATUSY AS ds
    WHERE ds.MISJA_ID_MOJE_FK = :misja_id
      AND ds.STATUS = N'0_ORYGINAŁ'
    GROUP BY ds.NPC_ID_FK
),

npc_zsumowane AS (
    SELECT
        NPC_ID,
        SUM(ILE_ZNAKOW) AS ILE_ZNAKOW
    FROM teksty_npc
    GROUP BY NPC_ID
)

SELECT
    n.RASA,
    SUM(nz.ILE_ZNAKOW) AS ILE_ZNAKOW
FROM npc_zsumowane AS nz
INNER JOIN dbo.NPC AS n
    ON n.NPC_ID_MOJE_PK = nz.NPC_ID
WHERE n.RASA IS NOT NULL
  AND n.RASA <> N'Unknown'
GROUP BY n.RASA
HAVING SUM(nz.ILE_ZNAKOW) > 30
ORDER BY ILE_ZNAKOW DESC;
""")

In [21]:
with silnik.connect() as conn:
    wybrane_rasy_list = conn.execute(q_select_rasa, {"misja_id": 535}).all()

In [26]:
wybrane_rasy = set(r[0] for r in wybrane_rasy_list)

In [27]:
wybrane_rasy

{'Haranir', 'Human', 'Orc'}

In [28]:
style_ras = list()
for rasa in wybrane_rasy:
    style_ras.append(RACE_STYLES[rasa])

In [ ]:
style_ras